## Proof of concept - IYKYK 
### (Personalized recommendations for restaurants, clubs and bars)

In [2]:
# !pip install openai pandas numpy python-dotenv

import os
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import pandas as pd

from openai import OpenAI

In [ ]:
# Environment setup, replace with your own API key

os.environ["OPENAI_API_KEY"] = "USE_YOUR_OWN_KEY_HERE"

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    #base_url=API_BASE_URL
)
CHAT_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"

### Dataset: 
For this PoC I will use a hardcoded set of venues in Zürich, with their corresponding vibes, descriptions and locations. 


In [4]:
# notice: we need more detailed lat and long, since all cities are in Zurich, this makes it seem like they are all at the same place. 
venues = [
    {
        "id": "v1",
        "name": "Les Halles",
        "type": "restaurant",
        "neighborhood": "Kreis 5",
        "vibe": "casual, lively, industrial, shared tables, afterwork crowd, relaxed",
        "opens": 17,
        "closes": 23,
        "lat": 47.385,
        "lon": 8.519,
    },
    {
        "id": "v2",
        "name": "Frau Gerolds Garten",
        "type": "restaurant",
        "neighborhood": "Kreis 5",
        "vibe": "outdoor, hip, colorful, garden, casual food, sunset drinks, apéro, social",
        "opens": 16,
        "closes": 23,
        "lat": 47.385,
        "lon": 8.518,
    },
    {
        "id": "v3",
        "name": "Kafi für Dich",
        "type": "restaurant",
        "neighborhood": "Kreis 4",
        "vibe": "cozy, warm, neighborhood, relaxed, date friendly, candlelight",
        "opens": 8,
        "closes": 23,
        "lat": 47.372,
        "lon": 8.529,
    },
    {
        "id": "v4",
        "name": "Soi Thai",
        "type": "restaurant",
        "neighborhood": "Kreis 4",
        "vibe": "spicy, thai food, casual, bustling, lively, colorful",
        "opens": 18,
        "closes": 23,
        "lat": 47.374,
        "lon": 8.525,
    },
    {
        "id": "v5",
        "name": "Raygrodski",
        "type": "bar",
        "neighborhood": "Kreis 4",
        "vibe": "cocktail bar, intimate, dim lights, creative drinks, date spot",
        "opens": 18,
        "closes": 2,
        "lat": 47.374,
        "lon": 8.526,
    },
    {
        "id": "v6",
        "name": "Longstreet Bar",
        "type": "bar",
        "neighborhood": "Kreis 4",
        "vibe": "queer friendly, loud, party, casual, mixed crowd, late night",
        "opens": 20,
        "closes": 4,
        "lat": 47.378,
        "lon": 8.527,
    },
    {
        "id": "v7",
        "name": "Plaza Klub",
        "type": "club",
        "neighborhood": "Kreis 4",
        "vibe": "club, dancing, commercial music, crowded, party",
        "opens": 23,
        "closes": 5,
        "lat": 47.375,
        "lon": 8.528,
    },
    {
        "id": "v8",
        "name": "Hive",
        "type": "club",
        "neighborhood": "Kreis 5",
        "vibe": "electronic music, techno, underground, late night, intense",
        "opens": 23,
        "closes": 6,
        "lat": 47.385,
        "lon": 8.519,
    },
    {
        "id": "v9",
        "name": "Bar 63",
        "type": "bar",
        "neighborhood": "Kreis 4",
        "vibe": "laid back, neighborhood bar, cheap drinks, students",
        "opens": 17,
        "closes": 1,
        "lat": 47.375,
        "lon": 8.53,
    },
    {
        "id": "v10",
        "name": "Gamper Restaurant",
        "type": "restaurant",
        "neighborhood": "Kreis 4",
        "vibe": "small plates, high quality, cozy but refined, foodie, natural wine",
        "opens": 18,
        "closes": 23,
        "lat": 47.375,
        "lon": 8.528,
    },
    {
        "id": "v11",
        "name": "Lupo",
        "type": "restaurant",
        "neighborhood": "Kreis 4",
        "vibe": "italian food, buzzing, lively, fancy, vibrant, friends gathering, sharing plates, date spot, natural wine",
        "opens": 16,
        "closes": 23,
        "lat": 47.3742609,
        "lon": 8.518,
    },
]

venues_df = pd.DataFrame(venues)
venues_df

,id,name,type,neighborhood,vibe,opens,closes,lat,lon
0,v1,Les Halles,restaurant,Kreis 5,"casual, lively, industrial, shared tables, aft...",17,23,47.385000,8.519
1,v2,Frau Gerolds Garten,restaurant,Kreis 5,"outdoor, hip, colorful, garden, casual food, s...",16,23,47.385000,8.518
2,v3,Kafi für Dich,restaurant,Kreis 4,"cozy, warm, neighborhood, relaxed, date friend...",8,23,47.372000,8.529
3,v4,Soi Thai,restaurant,Kreis 4,"spicy, thai food, casual, bustling, lively, co...",18,23,47.374000,8.525
4,v5,Raygrodski,bar,Kreis 4,"cocktail bar, intimate, dim lights, creative d...",18,2,47.374000,8.526
5,v6,Longstreet Bar,bar,Kreis 4,"queer friendly, loud, party, casual, mixed cro...",20,4,47.378000,8.527
6,v7,Plaza Klub,club,Kreis 4,"club, dancing, commercial music, crowded, party",23,5,47.375000,8.528
7,v8,Hive,club,Kreis 5,"electronic music, techno, underground, late ni...",23,6,47.385000,8.519
8,v9,Bar 63,bar,Kreis 4,"laid back, neighborhood bar, cheap drinks, stu...",17,1,47.375000,8.530
9,v10,Gamper Restaurant,restaurant,Kreis 4,"small plates, high quality, cozy but refined, ...",18,23,47.375000,8.528


### Embeddings

In [5]:
# helpers: 
def get_embedding(text: str) -> List[float]:
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp.data[0].embedding

# This only has to be done once for the list of venues
if "embedding" not in venues_df.columns:
    print("Computing embeddings for venues")
    embeddings = []
    for v in venues_df["vibe"].tolist():
        embeddings.append(get_embedding(v))
    venues_df["embedding"] = embeddings

# Here I am using a simple cosine similarity function, later this will be replaced by a vector DB query (FAISS probably), as well as reranking. 
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def search_venues_by_vibe(query: str, top_k: int = 5) -> pd.DataFrame:
    q_emb = np.array(get_embedding(query))
    sims = []
    for emb in venues_df["embedding"]:
        sims.append(cosine_sim(q_emb, np.array(emb)))
    venues_df["similarity"] = sims
    return venues_df.sort_values(by="similarity", ascending=False).head(top_k).copy()

Computing embeddings for venues


### LLM call helper: 

In [6]:
def chat(system_prompt: str, user_prompt: str, json_mode: bool = False) -> str:
    kwargs = {}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        **kwargs
    )
    return resp.choices[0].message.content


### Preference Agent 
Extract preferences and wishes from the natural language prompt of the user.

Input: Natural Language description from the user 

Output: Vibe, Preferred Location, Amount & Type of stops, Timeframe

In [7]:
import json

@dataclass
class Preferences:
    raw_request: str
    vibe_description: str
    neighborhoods: List[str]
    stops: List[str] # ["restaurant", "bar"] or ["bar"] or ["restaurant", "club"]
    earliest_start: int # hour in 24h format
    latest_end: int 

def run_preference_agent(user_request: str) -> Preferences:
    system_prompt = """
You are the Preference Agent for a Zurich nightlife planner.
You receive a natural-language request and must extract structured preferences.

Return ONLY valid JSON with keys:
- vibe_description: short summary of vibe they want
- neighborhoods: list of Zurich area names mentioned (e.g. ["Kreis 4", "Kreis 5", "Langstrasse"]), or [] if none
- stops: ordered list of types of venues they want to visit this night. Use only: "restaurant", "bar", "club".
- earliest_start: integer hour (0-23) when they want to start (guess from text if not explicit, default 18).
- latest_end: integer hour (0-23) when they want to be done (guess from text if not explicit, default 2).

Example:
{"vibe_description": "...", "neighborhoods": ["Kreis 4"], "stops": ["restaurant", "bar"], "earliest_start": 19, "latest_end": 1}
"""
    result = chat(system_prompt, user_request, json_mode=True)
    data = json.loads(result)

    return Preferences(
        raw_request=user_request,
        vibe_description=data.get("vibe_description", user_request),
        neighborhoods=data.get("neighborhoods", []),
        stops=data.get("stops", ["restaurant", "bar"]),
        earliest_start=int(data.get("earliest_start", 18)),
        latest_end=int(data.get("latest_end", 2)),
    )

prefs = run_preference_agent("I want a cozy dinner and then a lively bar near Kreis 4 around 19:30.")
prefs


Preferences(raw_request='I want a cozy dinner and then a lively bar near Kreis 4 around 19:30.', vibe_description='cozy dinner followed by a lively bar', neighborhoods=['Kreis 4'], stops=['restaurant', 'bar'], earliest_start=19, latest_end=2)

### Exploration Agent 

Takes the vibes description and the preferred neighborhoods and generates a list of candidate locations. 

In [8]:
def run_exploration_agent(prefs: Preferences, top_k_per_type: int = 5) -> Dict[str, pd.DataFrame]:
    """
    For each stop type (restaurant/bar/club) in prefs.stops,
    run a vibe-based search and lightly filter by neighborhood if provided.
    """
    results = {}
    base_query = prefs.vibe_description

    for stop_type in set(prefs.stops):
        df_candidates = search_venues_by_vibe(base_query, top_k=10)
        df_candidates = df_candidates[df_candidates["type"] == stop_type]

        if prefs.neighborhoods:
            # Simple filter (should later include lookup in nearby neighborhoods)
            mask = df_candidates["neighborhood"].apply(
                lambda n: any(n.lower() in nb.lower() or nb.lower() in n.lower()
                              for nb in prefs.neighborhoods)
            )
            df_candidates = df_candidates[mask] if mask.any() else df_candidates

        results[stop_type] = df_candidates.head(top_k_per_type)

    return results

# Test exploration:
candidates = run_exploration_agent(prefs)
candidates

{'restaurant':      id               name        type neighborhood  \
 2    v3      Kafi für Dich  restaurant      Kreis 4   
 10  v11               Lupo  restaurant      Kreis 4   
 9   v10  Gamper Restaurant  restaurant      Kreis 4   
 3    v4           Soi Thai  restaurant      Kreis 4   
 
                                                  vibe  opens  closes  \
 2   cozy, warm, neighborhood, relaxed, date friend...      8      23   
 10  italian food, buzzing, lively, fancy, vibrant,...     16      23   
 9   small plates, high quality, cozy but refined, ...     18      23   
 3   spicy, thai food, casual, bustling, lively, co...     18      23   
 
           lat    lon                                          embedding  \
 2   47.372000  8.529  [-0.01833494007587433, -0.025567634031176567, ...   
 10  47.374261  8.518  [-0.06566300243139267, -0.01700066402554512, -...   
 9   47.375000  8.528  [-0.019499478861689568, -0.022521067410707474,...   
 3   47.374000  8.525  [-0.017864

### Planning Agent 
 Given the preferences and the candidate list of places, pick one venue per stop, in the correct order. 

In [9]:
def format_candidates_for_llm(candidates: Dict[str, pd.DataFrame]) -> str:
    lines = []
    for stop_type, df in candidates.items():
        lines.append(f"\n=== {stop_type.upper()} CANDIDATES ===")
        for _, row in df.iterrows():
            lines.append(
                f"- id: {row['id']}, name: {row['name']}, neighborhood: {row['neighborhood']}, "
                f"vibe: {row['vibe']}, opens: {row['opens']}:00, closes: {row['closes']}:00, "
                f"lat: {row['lat']}, lon: {row['lon']}"
            )
    return "\n".join(lines)

def run_planning_agent(prefs: Preferences, candidates: Dict[str, pd.DataFrame]) -> Dict[str, Any]:
    system_prompt = """
You are the Planning Agent for a Zurich nightlife system.

You get:
- extracted preferences (stops, time window, neighborhoods, vibe)
- candidate venues per type (restaurant/bar/club)

You must choose an ordered list of concrete venues, one per stop in order.
Make sure the order makes sense with opening hours and rough time flow:
- dinner/restaurant earlier, then bar, then club etc.
Assume each stop takes ~2 hours.

Return ONLY JSON with:
- "plan": list of steps, each:
    {
      "order": 1,
      "stop_type": "restaurant" | "bar" | "club",
      "venue_id": "v3",
      "venue_name": "Kafi für Dich",
      "scheduled_start": 19,
      "scheduled_end": 21,
      "reason": "short explanation"
    }
- "notes": short free-text notes about your reasoning.
"""
    prefs_desc = f"""
Preferences:
- vibe: {prefs.vibe_description}
- neighborhoods: {prefs.neighborhoods}
- stops: {prefs.stops}
- earliest_start: {prefs.earliest_start}
- latest_end: {prefs.latest_end}
"""
    candidates_text = format_candidates_for_llm(candidates)
    user_prompt = prefs_desc + "\nCandidate venues:\n" + candidates_text

    result = chat(system_prompt, user_prompt, json_mode=True)
    return json.loads(result)

# Test planning:
plan = run_planning_agent(prefs, candidates)
plan

{'plan': [{'order': 1,
   'stop_type': 'restaurant',
   'venue_id': 'v3',
   'venue_name': 'Kafi für Dich',
   'scheduled_start': 19,
   'scheduled_end': 21,
   'reason': 'Cozy and warm vibe perfect for a relaxed dinner start in Kreis 4, fits the preference for a cozy dinner place, and open until 23:00.'},
  {'order': 2,
   'stop_type': 'bar',
   'venue_id': 'v5',
   'venue_name': 'Raygrodski',
   'scheduled_start': 21,
   'scheduled_end': 23,
   'reason': 'Lively cocktail bar with intimate ambiance and creative drinks, matching the preference for a lively bar after dinner, open until 2:00.'}],
 'notes': 'Selected Kafi für Dich for its cozy and warm vibe aligned with the dinner preference, located in Kreis 4 and open early. For the bar, Raygrodski offers a creative lively atmosphere suitable after a cozy dinner and also fits neighborhood and time constraints. Avoided clubs or late-night bars due to vibe and time window.'}

### Critique Agent

In [10]:
def get_venue_by_id(vid: str) -> Dict[str, Any]:
    row = venues_df[venues_df["id"] == vid]
    if row.empty:
        return {}
    return row.iloc[0].to_dict()

def run_critic_agent(plan: Dict[str, Any], prefs: Preferences) -> Dict[str, Any]:
    # constraints for the LLM
    steps = plan.get("plan", [])
    
    prefs_desc = f"""
Preferences:
- vibe: {prefs.vibe_description}
- neighborhoods: {prefs.neighborhoods}
- stops: {prefs.stops}
- earliest_start: {prefs.earliest_start}
- latest_end: {prefs.latest_end}
"""

    steps_text = "\n".join(
        f"- order {s.get('order')}: {s.get('stop_type')} at {s.get('venue_name')} "
        f"(id {s.get('venue_id')}), scheduled {s.get('scheduled_start')}-{s.get('scheduled_end')}"
        for s in steps
    )

    system_prompt = """
You are the Critic Agent for a Zurich nightlife planner.

You receive:
- user preferences (vibe, neighborhoods, time window)
- an ordered itinerary plan produced by another agent

Your tasks:
1. Check the plan for obvious issues:
   - times outside venue opening hours (assume times are roughly ok if no data)
   - schedule outside user's earliest_start / latest_end
   - obviously weird order (e.g. club before dinner)
   - too many stops, not matching requested sequence of types
2. If there are issues, produce a small fix by adjusting order or times or swapping venues.
3. Otherwise, keep the plan as-is.

OUTPUT FORMAT (VERY IMPORTANT):

Return ONLY valid JSON with exactly these keys at the top level:
- "accepted": boolean
- "issues": list of strings
- "revised_plan": list of steps

Each step in "revised_plan" MUST be an object with EXACTLY these keys:
- "order": integer
- "stop_type": string, one of "restaurant", "bar", "club"
- "venue_id": string
- "venue_name": string
- "scheduled_start": integer hour (0-23)
- "scheduled_end": integer hour (0-23)
- "reason": string

Do NOT use any alternative key names such as:
- "venue", "name", "id", "type", "start", "start_time", "end", "end_time", "scheduled"

Always use exactly:
- "venue_id", "venue_name", "stop_type", "scheduled_start", "scheduled_end", "order", "reason".

If you want to keep a step unchanged, copy it exactly but using these key names.
"""

    user_prompt = prefs_desc + "\n\nCurrent plan:\n" + steps_text

    raw = chat(system_prompt, user_prompt, json_mode=True)
    data = json.loads(raw)

    revised_plan = data.get("revised_plan", steps)

    return {
        "accepted": bool(data.get("accepted", True)),
        "issues": data.get("issues", []),
        "revised_plan": revised_plan,
    }

critique = run_critic_agent(plan, prefs)
critique


{'accepted': True,
 'issues': [],
 'revised_plan': [{'order': 1,
   'stop_type': 'restaurant',
   'venue_id': 'v3',
   'venue_name': 'Kafi für Dich',
   'scheduled_start': 19,
   'scheduled_end': 21,
   'reason': 'First stop is the cozy dinner as requested, starting at earliest start time.'},
  {'order': 2,
   'stop_type': 'bar',
   'venue_id': 'v5',
   'venue_name': 'Raygrodski',
   'scheduled_start': 21,
   'scheduled_end': 23,
   'reason': 'Second stop is the lively bar, correctly following the restaurant visit.'}]}

### End to End Pipeline

In [11]:
def run_pipeline(user_request: str) -> Dict[str, Any]:
    print("=== USER REQUEST ===")
    print(user_request)
    print()

    print("Running Preference Agent...")
    prefs = run_preference_agent(user_request)
    print(prefs)

    print("\n Running Exploration Agent...")
    candidates = run_exploration_agent(prefs)
    for t, df in candidates.items():
        print(f"\nTop candidates for {t}:")
        display(df[["id", "name", "neighborhood", "vibe", "opens", "closes", "similarity"]])

    print("\n Running Planning Agent...")
    plan = run_planning_agent(prefs, candidates)
    print(plan)

    print("\n Running Critic Agent...")
    critique = run_critic_agent(plan, prefs)
    print(critique)

    # If critic doesn't provide a revised_plan, use the original plan
    final_plan = critique.get("revised_plan") or plan.get("plan", [])

    print("\n=== FINAL ITINERARY ===")
    if not isinstance(final_plan, list):
        print("Final plan is not a list, got:", final_plan)
    else:
        print(final_plan)
        for idx, step in enumerate(final_plan, start=1):
            # be robust to weird outputs
            if not isinstance(step, dict):
                print(f"{idx}. [Unstructured step]:", step)
                continue

            order = step.get("order", idx)
            stop_type = step.get("stop_type", "stop")
            # handling different possible responses from LLM
            venue_name = step.get("venue_name", step.get("venue", step.get("venue_id", "Unknown venue")))
            s_start = step.get("scheduled_start", step.get("start","?"))
            s_end = step.get("scheduled_end", step.get("end", "?"))

            print(
                f"{order}. {stop_type.title()} – {venue_name} "
                f"from {s_start}:00 to {s_end}:00"
            )

    print("\nNotes and Issues:", critique.get("issues", []))

    return {
        "preferences": prefs,
        "candidates": candidates,
        "plan": plan,
        "critique": critique,
        "final_plan": final_plan,
    }

# Example run
_ = run_pipeline("I want a cute apéro spot to go to with my Boyfriend before dinner, the weather is so nice today. Then I am in the mood for some italian food.")


=== USER REQUEST ===
I want a cute apéro spot to go to with my Boyfriend before dinner, the weather is so nice today. Then I am in the mood for some italian food.

Running Preference Agent...
Preferences(raw_request='I want a cute apéro spot to go to with my Boyfriend before dinner, the weather is so nice today. Then I am in the mood for some italian food.', vibe_description='cute and romantic apéro spot followed by Italian dinner', neighborhoods=[], stops=['bar', 'restaurant'], earliest_start=18, latest_end=2)

 Running Exploration Agent...

Top candidates for restaurant:


,id,name,neighborhood,vibe,opens,closes,similarity
10,v11,Lupo,Kreis 4,"italian food, buzzing, lively, fancy, vibrant,...",16,23,0.621596
9,v10,Gamper Restaurant,Kreis 4,"small plates, high quality, cozy but refined, ...",18,23,0.523500
1,v2,Frau Gerolds Garten,Kreis 5,"outdoor, hip, colorful, garden, casual food, s...",16,23,0.518669
2,v3,Kafi für Dich,Kreis 4,"cozy, warm, neighborhood, relaxed, date friend...",8,23,0.488980
0,v1,Les Halles,Kreis 5,"casual, lively, industrial, shared tables, aft...",17,23,0.424881



Top candidates for bar:


,id,name,neighborhood,vibe,opens,closes,similarity
4,v5,Raygrodski,Kreis 4,"cocktail bar, intimate, dim lights, creative d...",18,2,0.546862
5,v6,Longstreet Bar,Kreis 4,"queer friendly, loud, party, casual, mixed cro...",20,4,0.343774
8,v9,Bar 63,Kreis 4,"laid back, neighborhood bar, cheap drinks, stu...",17,1,0.340371



 Running Planning Agent...
{'plan': [{'order': 1, 'stop_type': 'bar', 'venue_id': 'v5', 'venue_name': 'Raygrodski', 'scheduled_start': 18, 'scheduled_end': 20, 'reason': 'Raygrodski is a cute, intimate cocktail bar with a romantic vibe and creative drinks, perfect for a charming apéro start from earliest start time 18.'}, {'order': 2, 'stop_type': 'restaurant', 'venue_id': 'v11', 'venue_name': 'Lupo', 'scheduled_start': 20, 'scheduled_end': 22, 'reason': 'Lupo offers authentic Italian food with a lively and vibrant atmosphere suited for a romantic Italian dinner following the apéro spot, open until 23:00 so starting at 20 leaves good time.'}], 'notes': "The plan respects the time flow by starting with a cute and romantic cocktail bar as apéro around 18. Raygrodski's intimate vibe fits the requested cute and romantic apéro spot. Following this with Lupo as an Italian restaurant for dinner ensures the Italian preference is met in a lively, date-friendly setting. Both venues are in Kreis

## Things I want to add / improve: 
- Obviously not hard code the list of places
- But also fetch & create the vibes dynamically
  - Initially the vibes should be scraped (chatGPT can generate a description)
  - Later, users should be able to add their own descriptions, and we should periodically recompute the vibes
  - Even better: give people the ability to upload pictures of the evening (or make this required?), so that we can infer the vibe through CNN / other visual models.
- In addition to the vibe embedding matching, we should learn the users actual preferences 
  - Here it will be interesting to investigate how much data is required for the recommendations to be meaningful, 
  - And what kind of latent preferences we can learn
  - It would also be very interesting to see if we can learn latent preferences from the users digital footprint => their insta, their youtube etc. (if we can get access to this)
- Also we need to access an actual map API (in my frontend, we currently use MapBox), that has access to data about public transport => how do people get from A to B? 
- Maybe even fine tune model to each office. This is not feasible for single people, but if its a small company, it should be feasible. Make it learn the decisions of the traders of this particular office. => learn more about fine tuning and the opportunities 